In [ ]:
# !pip install shap
# %pip install -U google-genai

In [47]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from imblearn.combine import SMOTEENN
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import shap
import os
import joblib
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [48]:
# from google import genai

# client = genai.Client(
#     api_key=os.getenv("GEMINI_API_KEY")
# )

# print("Gemini client created successfully!")

# response = client.models.generate_content(
#     model="gemini-3.1-flash-lite",
#     contents="Explain customer churn prediction in simple terms."
# )

In [49]:
df = pd.read_excel("../data/data_cleaned.xlsx")

In [50]:
df.shape

(7032, 29)

In [51]:
df.columns.to_list()

['City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'CLTV',
 'Total Services',
 'New Customer',
 'Avg Monthly Spend']

In [52]:
location_cols = [
    'City',
    'Zip Code',
    'Lat Long',
    'Latitude',
    'Longitude',
    'CLTV'
]

df.drop(columns=location_cols, inplace=True)

In [53]:
df['Services_per_Month'] = (df['Total Services'] / df['Tenure Months'].replace(0,1))

In [54]:
df["High_Value"] = (
    df["Monthly Charges"] >
    df["Monthly Charges"].median()
)
df['High_Value'] = df['High_Value'].astype(int)
df['High_Value'].head()

0    0
1    1
2    1
3    1
4    1
Name: High_Value, dtype: int64

In [55]:
security_cols = [
    "Online Security",
    "Device Protection",
    "Tech Support"
]

df["Security Bundle"] = (
    (df["Online Security"] == "Yes").astype(int) +
    (df["Device Protection"] == "Yes").astype(int) +
    (df["Tech Support"] == "Yes").astype(int)
)

In [56]:
df["Entertainment Bundle"] = (
    (df["Streaming TV"] == "Yes").astype(int) +
    (df["Streaming Movies"] == "Yes").astype(int)
)

In [57]:
df["Long-Term Customer"] = (
    df["Tenure Months"] > 24
).astype(int)

In [58]:
df[
    [
        "Security Bundle",
        "Entertainment Bundle",
        "Long-Term Customer"
    ]
].head()

,Security Bundle,Entertainment Bundle,Long-Term Customer
0,1,0,0
1,0,0,0
2,1,2,0
3,2,2,1
4,1,2,1


In [59]:
df.columns.to_list()

['Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Total Services',
 'New Customer',
 'Avg Monthly Spend',
 'Services_per_Month',
 'High_Value',
 'Security Bundle',
 'Entertainment Bundle',
 'Long-Term Customer']

In [60]:
# Features and Target
X = df.drop(columns='Churn Label')
y = df['Churn Label']

In [ ]:
print("Features:", X.shape)
print("Target:", y.shape)

Features: (7032, 27)
Target: (7032,)


In [62]:
y.value_counts()

Churn Label
No     5163
Yes    1869
Name: count, dtype: int64

In [63]:
y = y.map({
    'No': 0,
    'Yes': 1
})

In [65]:
df.shape

(7032, 28)

In [64]:
# output_file = "data_cleaned_featued.xlsx"
# df.to_excel(output_file, index=False)

# print("Cleaned dataset saved as:", output_file)

Cleaned dataset saved as: data_cleaned_featued.xlsx


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTraining Churn Distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting Churn Distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
X.dtypes

In [ ]:
numeric = ['int64', 'float64']
numeric_features = X.select_dtypes(include = numeric).columns

categorical_features = X.select_dtypes(include = ['object',  'category']).columns

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print(X_train_processed.shape)
print(X_test_processed.shape)

In [ ]:
data = pd.DataFrame(X_train_processed)
data.head()

# **MODEL TRAINING**

# ***Logistic Regression***

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [ ]:
log_model.fit(X_train_processed, y_train)

In [ ]:
y_pred = log_model.predict(X_test_processed)
y_prob = log_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Logistic Regression Results")
print("---------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

*The Logistic Regression baseline achieved an accuracy of 79.96% and a ROC-AUC of 0.842, demonstrating good overall discriminatory performance. However, the recall for the churn class was 56.68%, indicating that the model missed a considerable proportion of actual churners. Therefore, additional models and optimization techniques will be evaluated to improve churn detection.*

# **FINE TUNING LOGISTIC REGRESSION MODEL**

In [ ]:
param_grid = [
    {
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga'],
        'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    },
    {
        'penalty': ['l2'],
        'solver': ['lbfgs', 'newton-cg', 'sag'],
        'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        'max_iter': [100, 250, 500, 1000]
    },
    {
        'penalty': ['elasticnet'],
        'solver': ['saga'],
        'C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'l1_ratio': [0.1, 0.5, 0.9],
        'max_iter': [100, 250, 500, 1000]
    }
]

In [ ]:
grid_search = GridSearchCV(
    estimator=log_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    return_train_score = False
)

grid_search.fit(X_train_processed, y_train)
# grid_search.cv_results_

In [ ]:
grid_df = pd.DataFrame(grid_search.cv_results_)
# grid_df.head()

In [ ]:
grid_df[
    [
        "param_C",
        "param_penalty",
        "param_solver",
        "param_max_iter",
        "param_l1_ratio",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score").head()

In [ ]:
grid_search.best_params_

*BEST PARAMETERS FOR LOGISTIC REGRESSION*

{'C': 10.0, 'penalty': 'l1', 'solver': 'liblinear'}

In [ ]:
best_log_model = grid_search.best_estimator_

In [ ]:
y_pred_tuned = best_log_model.predict(X_test_processed)
y_prob_tuned = best_log_model.predict_proba(X_test_processed)[:, 1]
y_prob_tuned

In [ ]:
accuracy_flog = accuracy_score(y_test, y_pred_tuned)
precision_flog = precision_score(y_test, y_pred_tuned)
recall_flog = recall_score(y_test, y_pred_tuned)
f1_flog = f1_score(y_test, y_pred_tuned)
roc_auc_flong = roc_auc_score(y_test, y_prob_tuned)

print("Tuned Logistic Regression Results")
print("---------------------")
print("Accuracy :", round(accuracy_flog, 4))
print("Precision:", round(precision_flog, 4))
print("Recall   :", round(recall_flog, 4))
print("F1 Score :", round(f1_flog, 4))
print("ROC-AUC  :", round(roc_auc_flong, 4))

print(classification_report(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_tuned))

## **THRESHOLDED**

In [ ]:
thresholds = np.arange(0.2, 0.81, 0.01)

best_f1 = 0
best_threshold = 0

for t in thresholds:
    preds = (y_prob_tuned >= t).astype(int)
    score = f1_score(y_test, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

final_preds = (y_prob_tuned >= best_threshold).astype(int)
print(best_threshold)
print("Tuned and Thresholded Logistic Regression Results")
print("---------------------")
print("Accuracy :", accuracy_score(y_test, final_preds))
print("Precision:", precision_score(y_test, final_preds))
print("Recall   :", recall_score(y_test, final_preds))
print("F1 Score :", f1_score(y_test, final_preds))
print(classification_report(y_test, final_preds))
# print("ROC-AUC:", roc_auc_score(y_test, y_prob_tuned))

# **SMOTE**

In [ ]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=100)

X_train_resampled, y_train_resampled = sm.fit_resample(
    X_train_processed,
    y_train
)

log_smote = LogisticRegression()
log_smote.fit(X_train_resampled, y_train_resampled)

In [ ]:
y_pred = best_log_model.predict(X_test_processed)
y_prob = best_log_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
accuracy_flog = accuracy_score(y_test, y_pred)
precision_flog = precision_score(y_test, y_pred)
recall_flog = recall_score(y_test, y_pred)
f1_flog = f1_score(y_test, y_pred)
roc_auc_flong = roc_auc_score(y_test, y_prob)

print("SMOTE Logistic Regression Results")
print("---------------------")
print("Accuracy :", round(accuracy_flog, 4))
print("Precision:", round(precision_flog, 4))
print("Recall   :", round(recall_flog, 4))
print("F1 Score :", round(f1_flog, 4))
print("ROC-AUC  :", round(roc_auc_flong, 4))

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

## ***Decision Tree Classifier***

In [ ]:
dt_model = DecisionTreeClassifier(
    random_state=100,
    max_depth=6, 
    min_samples_leaf=8
)

dt_model.fit(X_train_processed, y_train)

y_pred_dt = dt_model.predict(X_test_processed)
y_prob_dt = dt_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt)

print("Decision Tree Results")
print("---------------------")
print("Accuracy :", round(accuracy_dt, 4))
print("Precision:", round(precision_dt, 4))
print("Recall   :", round(recall_dt, 4))
print("F1 Score :", round(f1_dt, 4))
print("ROC-AUC  :", round(roc_auc_dt, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

# **FINE TUNING DECISION TREE CLASSIFIER**

In [ ]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'ccp_alpha': [0.0, 0.001]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    return_train_score = False
)

grid_search.fit(X_train_processed, y_train)

In [ ]:
grid_dt = pd.DataFrame(grid_search.cv_results_)
# grid_dt.sort_values("rank_test_score").head(5)

In [ ]:
grid_search.best_params_

*BEST PARAMETERS FOR DECISION TREE*

{'ccp_alpha': 0.001,
 'criterion': 'gini',
 'max_depth': 5,
 'max_features': None,
 'min_samples_leaf': 1,
 'min_samples_split': 2}

In [ ]:
best_dt_model = grid_search.best_estimator_

In [ ]:
y_pred_tuned = best_dt_model.predict(X_test_processed)

y_prob_tuned = best_dt_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
accuracy_rf_tuned = accuracy_score(y_test, y_pred_tuned)
precision_rf_tuned = precision_score(y_test, y_pred_tuned)
recall_rf_tuned = recall_score(y_test, y_pred_tuned)
f1_rf_tuned = f1_score(y_test, y_pred_tuned)
roc_auc_rf_tuned = roc_auc_score(y_test, y_prob_tuned)

print("Decision Tree Results")
print("---------------------")
print("Accuracy :", round(accuracy_rf_tuned, 4))
print("Precision:", round(precision_rf_tuned, 4))
print("Recall   :", round(recall_rf_tuned, 4))
print("F1 Score :", round(f1_rf_tuned, 4))
print("ROC-AUC  :", round(roc_auc_rf_tuned, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

# **SMOTEEN IT FURTHER**

In [ ]:
sm = SMOTEENN()

X_train_resampled1, y_train_resampled1 = sm.fit_resample(
    X_train_processed,
    y_train
)

xr_train1,xr_test1,yr_train1,yr_test1=train_test_split(X_train_resampled1, y_train_resampled1,test_size=0.2)

model_dt_smote=DecisionTreeClassifier(criterion = "gini",random_state = 100,max_depth=6, min_samples_leaf=8)

# 4. Train the model
model_dt_smote.fit(X_train_resampled1, y_train_resampled1)

# 5. Evaluate on the untouched test set
y_predict = model_dt_smote.predict(X_test_processed)
y_probab = model_dt_smote.predict_proba(X_test_processed)[:, 1]

In [ ]:
print("Decision Tree Results (SMOTEENN)")
print("---------------------")
print("Accuracy :", accuracy_score(y_test, y_predict))
print("Precision:", precision_score(y_test, y_predict))
print("Recall   :", recall_score(y_test, y_predict))
print("F1 Score :", f1_score(y_test, y_predict))
print("ROC-AUC  :", roc_auc_score(y_test, y_probab))

print("\nClassification Report:")
print(classification_report(y_test, y_predict))

## ***Random Forest***

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    random_state = 100,
    max_depth=6, 
    min_samples_leaf=8
)

rf_model.fit(X_train_processed, y_train)

y_pred_rf = rf_model.predict(X_test_processed)
y_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", round(accuracy_rf, 4))
print("Precision:", round(precision_rf, 4))
print("Recall   :", round(recall_rf, 4))
print("F1 Score :", round(f1_rf, 4))
print("ROC-AUC  :", round(roc_auc_rf, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

*Among the three evaluated models, Logistic Regression achieved the best overall performance, with an accuracy of 79.96% and ROC-AUC of 0.842. It also achieved the highest recall (56.68%) and F1-score (60.06%) for the churn class, indicating a stronger ability to identify customers at risk of churning. Decision Tree and Random Forest models showed comparatively lower performance across the evaluated metrics. Therefore, Logistic Regression was selected as the initial best-performing model.*

# **FINE TUNING RANDOM FOREST MODEL**

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt'],
    'bootstrap': [True]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    return_train_score = False
)

grid_search.fit(X_train_processed, y_train)

In [ ]:
grid_rf = pd.DataFrame(grid_search.cv_results_)
# grid_rf.sort_values("rank_test_score").head(10)

In [ ]:
grid_search.best_params_

*BEST PARAMETERS FOR RANDOM FOREST*

{'bootstrap': True,
 'max_depth': 10,
 'max_features': 'sqrt',
 'min_samples_leaf': 2,
 'min_samples_split': 5,
 'n_estimators': 100}

In [ ]:
best_rf_model = grid_search.best_estimator_

In [ ]:
y_pred_tuned = best_rf_model.predict(X_test_processed)

y_prob_tuned = best_rf_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
accuracy_rf_tuned = accuracy_score(y_test, y_pred_tuned)
precision_rf_tuned = precision_score(y_test, y_pred_tuned)
recall_rf_tuned = recall_score(y_test, y_pred_tuned)
f1_rf_tuned = f1_score(y_test, y_pred_tuned)
roc_auc_rf_tuned = roc_auc_score(y_test, y_prob_tuned)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", round(accuracy_rf_tuned, 4))
print("Precision:", round(precision_rf_tuned, 4))
print("Recall   :", round(recall_rf_tuned, 4))
print("F1 Score :", round(f1_rf_tuned, 4))
print("ROC-AUC  :", round(roc_auc_rf_tuned, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

## **SMOTEENN IT FURTHER**

In [ ]:
sm = SMOTEENN()

X_train_resampled, y_train_resampled = sm.fit_resample(
    X_train_processed,
    y_train
)

xr_train1,xr_test1,yr_train1,yr_test1=train_test_split(X_train_resampled, y_train_resampled,test_size=0.2)

model_rf_smote=RandomForestClassifier(bootstrap= True, n_estimators=100, random_state = 100, criterion='gini', max_depth=10, min_samples_leaf=1, max_features= 'sqrt', min_samples_split= 2)

# 4. Train the model
model_rf_smote.fit(X_train_resampled, y_train_resampled)

# 5. Evaluate on the untouched test set
y_pred = model_rf_smote.predict(X_test_processed)
y_prob = model_rf_smote.predict_proba(X_test_processed)[:, 1]

In [ ]:
print("Random Forest Results (SMOOTHEN)")
print("---------------------")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ***XGBOOST***

In [ ]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train_processed, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test_processed)
y_prob_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
print("XGBoost Results")
print("---------------------")

print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall   :", recall_score(y_test, y_pred_xgb))
print("F1 Score :", f1_score(y_test, y_pred_xgb))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train_processed, y_train)

best_xgb_model = grid_search.best_estimator_

print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
best_xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    learning_rate=0.05,
    n_estimators=200,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8
)

In [ ]:
best_xgb_model.fit(X_train_processed, y_train)

In [ ]:
y_pred = best_xgb_model.predict(X_test_processed)
y_prob = best_xgb_model.predict_proba(X_test_processed)[:,1]

print("Tuned XGBoost Results")
print("---------------------")

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print(classification_report(y_test, y_pred))

In [ ]:
thresholds = np.arange(0.2,0.81,0.01)

best_f1 = 0
best_threshold = 0

for t in thresholds:
    preds = (y_prob >= t).astype(int)
    score = f1_score(y_test, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

final_preds = (y_prob >= best_threshold).astype(int)

In [ ]:
print("\nThreshold Optimized XGBoost Results")
print("-----------------------------------")
print("Accuracy :", accuracy_score(y_test, final_preds))
print("Precision:", precision_score(y_test, final_preds))
print("Recall   :", recall_score(y_test, final_preds))
print("F1 Score :", f1_score(y_test, final_preds))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, final_preds))

## **COMPAIRING THE MODELS**

In [ ]:
results = [
    {
        "Model": "Logistic Regression",
        "Variant": "Baseline",
        "Accuracy": 0.7967,
        "Precision": 0.6279,
        "Recall": 0.5775,
        "F1": 0.6017,
        "ROC-AUC": 0.8447
    },
    {
        "Model": "Logistic Regression",
        "Variant": "Hyperparameter Tuned",
        "Accuracy": 0.7946,
        "Precision": 0.6246,
        "Recall": 0.5695,
        "F1": 0.5958,
        "ROC-AUC": 0.8443
    },
    {
        "Model": "Logistic Regression",
        "Variant": "SMOTE",
        "Accuracy": 0.7306,
        "Precision": 0.4958,
        "Recall": 0.7941,
        "F1": 0.6105,
        "ROC-AUC": 0.8422
    },
    {
        "Model": "Logistic Regression",
        "Variant": "Threshold Optimized",
        "Accuracy": 0.8003,
        "Precision": 0.6177,
        "Recall": 0.6524,
        "F1": 0.6346,
        "ROC-AUC": 0.8443
    },

    {
        "Model": "Decision Tree",
        "Variant": "Baseline",
        "Accuracy": 0.7875,
        "Precision": 0.5984,
        "Recall": 0.6096,
        "F1": 0.6040,
        "ROC-AUC": 0.8348
    },
    {
        "Model": "Decision Tree",
        "Variant": "Hyperparameter Tuned",
        "Accuracy": 0.7868,
        "Precision": 0.6000,
        "Recall": 0.5936,
        "F1": 0.5968,
        "ROC-AUC": 0.8220
    },
    {
        "Model": "Decision Tree",
        "Variant": "Hyperparameter + SMOTEENN",
        "Accuracy": 0.7029,
        "Precision": 0.4666,
        "Recall": 0.8209,
        "F1": 0.5950,
        "ROC-AUC": 0.8003
    },
    {
        "Model": "Random Forest",
        "Variant": "Baseline",
        "Accuracy": 0.7918,
        "Precision": 0.6494,
        "Recall": 0.4706,
        "F1": 0.5457,
        "ROC-AUC": 0.8452
    },
    {
        "Model": "Random Forest",
        "Variant": "Hyperparameter + SMOTEENN",
        "Accuracy": 0.7214,
        "Precision": 0.4857,
        "Recall": 0.8155,
        "F1": 0.6088,
        "ROC-AUC": 0.8382
    },
    {
        "Model": "XGBoost",
        "Variant": "Baseline",
        "Accuracy": 0.7953091684434968,
        "Precision": 0.634375,
        "Recall": 0.5427807486631016,
        "F1": 0.5850144092219021,
        "ROC-AUC": 0.8478938349959362
    },
    {   
        "Model": "XGBoost",
        "Variant": "Threshold Optimized",
        "Accuracy": 0.7846481876332623,
        "Precision": 0.5773420479302832,
        "Recall": 0.7085561497326203,
        "F1": 0.6362545018007203,
        "ROC-AUC": 0.8478938349959362
    }
]

results_df = pd.DataFrame(results)
results_df

#### **Model Selection: Multiple machine learning models, including Logistic Regression, Decision Tree, Random Forest, and XGBoost, were evaluated using baseline, hyperparameter tuning, resampling techniques (SMOTE/SMOTEENN), and threshold optimization. Among all the evaluated models, Threshold-Optimized Logistic Regression was selected as the final model as it provided the best balance between predictive performance and interpretability. It achieved an accuracy of 80.03%, precision of 61.77%, recall of 65.24%, F1-score of 63.46%, and a ROC-AUC of 84.43%. Although Threshold-Optimized XGBoost produced a marginally higher F1-score (63.63%) and ROC-AUC (84.79%), the improvement was minimal. Logistic Regression offered comparable performance while remaining computationally efficient, easier to interpret, and well-suited for generating explainable predictions using SHAP. Therefore, Threshold-Optimized Logistic Regression was selected as the final model for the customer churn prediction system.**

## **Production Pipeline for Flask Deployment**

In [ ]:
production_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", best_log_model)
    ]
)

In [ ]:
production_pipeline.fit(X_train, y_train)

In [ ]:
threshold = 0.45000000000000023 

y_prob_pipeline = production_pipeline.predict_proba(X_test)[:, 1]
y_pred_pipeline = (y_prob_pipeline >= threshold).astype(int)

In [ ]:
accuracy = accuracy_score(y_test, y_pred_pipeline)
precision = precision_score(y_test, y_pred_pipeline)
recall = recall_score(y_test, y_pred_pipeline)
f1 = f1_score(y_test, y_pred_pipeline)
roc_auc = roc_auc_score(y_test, y_prob_pipeline)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred_pipeline))

# print("\nConfusion Matrix")
# print(confusion_matrix(y_test, y_pred_pipeline))

In [ ]:
os.makedirs("../model", exist_ok=True)

joblib.dump(
    production_pipeline,
    "../model/churn_pipeline.pkl"
)

print("✅ Production pipeline saved successfully!")

In [ ]:
joblib.dump(best_threshold, "../model/threshold.pkl")

## **SHAP EXPLANATION**

In [ ]:
# feature_names = preprocessor.get_feature_names_out()
# print(feature_names)

In [ ]:
# joblib.dump(
#     feature_names,
#     "../model/feature_names.pkl"
# )

In [ ]:
# print(type(X_train_processed))
# print(X_train_processed.shape)
# print(len(feature_names))

In [ ]:
# explainer = shap.LinearExplainer(
#     best_log_model,
#     X_train_processed
# )

In [ ]:
# shap_values = explainer.shap_values(X_test_processed)

In [ ]:
# print(shap_values.shape)

In [ ]:
# shap.summary_plot(
#     shap_values,
#     X_test_processed,
#     feature_names=feature_names
# )

In [ ]:
# shap.summary_plot(
#     shap_values,
#     X_test_processed,
#     feature_names=feature_names,
#     plot_type='bar'
# )

In [ ]:
# shap.waterfall_plot(
#     shap.Explanation(
#         values=shap_values[5],
#         base_values=explainer.expected_value,
#         data=X_test_processed[5],
#         feature_names=feature_names
#     )
# )

In [ ]:
# os.makedirs("../model", exist_ok=True)

# joblib.dump(
#     X_train_processed,
#     "../model/shap_background.pkl"
# )

# print("✅ SHAP background dataset saved successfully!")

In [ ]:
# for col in categorical_features:
#     print(f"\n{col}")
#     print(sorted(df[col].unique()))

In [ ]:
# feature_name_mapping = {
#     # Numerical features
#     'num__Tenure Months': 'Customer Tenure',
#     'num__Monthly Charges': 'Monthly Charges',
#     'num__Total Charges': 'Total Charges',
#     'num__CLTV': 'Customer Lifetime Value',
#     'num__Total Services': 'Number of Services',
#     'num__New Customer': 'New Customer',
#     'num__Avg Monthly Spend': 'Average Monthly Spend',

#     # Demographics
#     'cat__Gender_Female': 'Female',
#     'cat__Gender_Male': 'Male',
#     'cat__Senior Citizen_No': 'Not a Senior Citizen',
#     'cat__Senior Citizen_Yes': 'Senior Citizen',
#     'cat__Partner_No': 'No Partner',
#     'cat__Partner_Yes': 'Has Partner',
#     'cat__Dependents_No': 'No Dependents',
#     'cat__Dependents_Yes': 'Has Dependents',

#     # Phone Services
#     'cat__Phone Service_No': 'No Phone Service',
#     'cat__Phone Service_Yes': 'Phone Service',

#     # Multiple Lines
#     'cat__Multiple Lines_No': 'No Multiple Lines',
#     'cat__Multiple Lines_No Phone Service': 'No Phone Service',
#     'cat__Multiple Lines_Yes': 'Multiple Lines',

#     # Internet Service
#     'cat__Internet Service_Dsl': 'DSL Internet',
#     'cat__Internet Service_Fiber Optic': 'Fiber Optic Internet',
#     'cat__Internet Service_No': 'No Internet Service',

#     # Online Security
#     'cat__Online Security_No': 'No Online Security',
#     'cat__Online Security_No Internet Service': 'No Internet Service',
#     'cat__Online Security_Yes': 'Online Security',

#     # Online Backup
#     'cat__Online Backup_No': 'No Online Backup',
#     'cat__Online Backup_No Internet Service': 'No Internet Service',
#     'cat__Online Backup_Yes': 'Online Backup',

#     # Device Protection
#     'cat__Device Protection_No': 'No Device Protection',
#     'cat__Device Protection_No Internet Service': 'No Internet Service',
#     'cat__Device Protection_Yes': 'Device Protection',

#     # Tech Support
#     'cat__Tech Support_No': 'No Tech Support',
#     'cat__Tech Support_No Internet Service': 'No Internet Service',
#     'cat__Tech Support_Yes': 'Tech Support',

#     # Streaming TV
#     'cat__Streaming TV_No': 'No Streaming TV',
#     'cat__Streaming TV_No Internet Service': 'No Internet Service',
#     'cat__Streaming TV_Yes': 'Streaming TV',

#     # Streaming Movies
#     'cat__Streaming Movies_No': 'No Streaming Movies',
#     'cat__Streaming Movies_No Internet Service': 'No Internet Service',
#     'cat__Streaming Movies_Yes': 'Streaming Movies',

#     # Contract
#     'cat__Contract_Month-To-Month': 'Month-to-Month Contract',
#     'cat__Contract_One Year': 'One-Year Contract',
#     'cat__Contract_Two Year': 'Two-Year Contract',

#     # Billing
#     'cat__Paperless Billing_No': 'No Paperless Billing',
#     'cat__Paperless Billing_Yes': 'Paperless Billing',

#     # Payment Method
#     'cat__Payment Method_Bank Transfer (Automatic)': 'Automatic Bank Transfer',
#     'cat__Payment Method_Credit Card (Automatic)': 'Automatic Credit Card',
#     'cat__Payment Method_Electronic Check': 'Electronic Check',
#     'cat__Payment Method_Mailed Check': 'Mailed Check'
# }

In [ ]:
# def generate_recommendations(
#     churn_probability,
#     risk_factors
# ):
    
#     recommendations = []

#     # High churn risk
#     if churn_probability >= 0.7:
#         recommendations.append(
#             "Prioritize this customer for immediate retention outreach."
#         )

#     # Medium churn risk
#     elif churn_probability >= 0.4:
#         recommendations.append(
#             "Monitor this customer closely and consider a targeted retention offer."
#         )

#     # Low churn risk
#     else:
#         recommendations.append(
#             "No immediate retention intervention is required. Continue regular engagement."
#         )

#     # Check individual risk factors
#     for _, row in risk_factors.iterrows():

#         feature = row['Feature']

#         if feature == 'Month-to-Month Contract':
#             recommendations.append(
#                 "Offer an incentive to upgrade to a one-year or two-year contract."
#             )

#         elif feature == 'Monthly Charges':
#             recommendations.append(
#                 "Review the customer's monthly charges and consider a personalized discount or value-added offer."
#             )

#         elif feature == 'Average Monthly Spend':
#             recommendations.append(
#                 "Evaluate whether the customer's spending is perceived as high relative to the services received and consider a value-based offer."
#             )

#         elif feature == 'No Online Security':
#             recommendations.append(
#                 "Offer an online security package or bundle to increase perceived service value."
#             )

#         elif feature == 'No Tech Support':
#             recommendations.append(
#                 "Offer a technical support package or complimentary support period."
#             )

#         elif feature == 'Fiber Optic Internet':
#             recommendations.append(
#                 "Review the customer's fiber-optic service experience, pricing, and satisfaction."
#             )

#         elif feature == 'Electronic Check':
#             recommendations.append(
#                 "Encourage the customer to switch to an automatic payment method and offer a small billing incentive if appropriate."
#             )

#         elif feature == 'No Dependents':
#             recommendations.append(
#                 "Consider personalized engagement offers that emphasize flexibility and individual customer value."
#             )

#         elif feature == 'Streaming TV':
#             recommendations.append(
#                 "Review streaming service usage and consider bundling entertainment services with the customer's plan."
#             )

#         elif feature == 'Streaming Movies':
#             recommendations.append(
#                 "Consider offering a bundled entertainment package to increase perceived value."
#             )

#         elif feature == 'New Customer':
#             recommendations.append(
#                 "Strengthen early-stage customer engagement with onboarding support and proactive follow-up."
#             )

#     return recommendations

In [ ]:
# def analyze_customer(customer_index):

#     customer_data = X_test_processed[customer_index]

#     # Reshape for model prediction
#     customer_data_2d = customer_data.reshape(1, -1)

#     churn_probability = best_log_model.predict_proba(
#         customer_data_2d
#     )[0, 1]

#     prediction = best_log_model.predict(
#         customer_data_2d
#     )[0]

#     if churn_probability >= 0.7:
#         risk_level = 'HIGH'

#     elif churn_probability >= 0.4:
#         risk_level = 'MEDIUM'

#     else:
#         risk_level = 'LOW'


#     customer_shap = shap_values[customer_index]

#     explanation = pd.DataFrame({
#         'Feature': feature_names,
#         'SHAP Value': customer_shap,
#         'Feature Value': customer_data
#     })

#     explanation = explanation[
#         explanation['Feature Value'] != 0
#     ].copy()

#     explanation['Feature'] = explanation['Feature'].map(
#         feature_name_mapping
#     ).fillna(explanation['Feature'])

#     risk_factors = (
#         explanation[
#             explanation['SHAP Value'] > 0
#         ]
#         .sort_values(
#             by='SHAP Value',
#             ascending=False
#         )
#         .head(5)
#     )

#     protective_factors = (
#         explanation[
#             explanation['SHAP Value'] < 0
#         ]
#         .sort_values(
#             by='SHAP Value'
#         )
#         .head(5)
#     )

#     recommendations = []


#     # HIGH RISK
#     if risk_level == 'HIGH':

#         recommendations.append(
#             "Prioritize this customer for immediate retention outreach."
#         )

#         recommendations.append(
#             "Consider offering a personalized retention incentive."
#         )


#     # MEDIUM RISK
#     elif risk_level == 'MEDIUM':

#         recommendations.append(
#             "Monitor this customer closely for changes in engagement."
#         )

#         recommendations.append(
#             "Consider a targeted retention offer based on the customer's risk factors."
#         )


#     # LOW RISK
#     else:

#         recommendations.append(
#             "No immediate retention intervention is required."
#         )

#         recommendations.append(
#             "Continue regular customer engagement and monitor future behavior."
#         )

#     for _, row in risk_factors.iterrows():

#         feature = row['Feature']


#         if feature == 'Month-to-Month Contract':

#             recommendations.append(
#                 "Consider offering an incentive to upgrade to a longer-term contract."
#             )


#         elif feature == 'Monthly Charges':

#             recommendations.append(
#                 "Review the customer's monthly charges and consider a personalized value-based offer."
#             )


#         elif feature == 'Average Monthly Spend':

#             recommendations.append(
#                 "Evaluate whether the customer's spending is high relative to the services received and consider improving perceived value."
#             )


#         elif feature == 'No Online Security':

#             recommendations.append(
#                 "Consider offering an online security package or bundle."
#             )


#         elif feature == 'No Tech Support':

#             recommendations.append(
#                 "Consider offering a technical support package or complimentary support period."
#             )


#         elif feature == 'Fiber Optic Internet':

#             recommendations.append(
#                 "Review the customer's fiber-optic service experience, pricing, and satisfaction."
#             )


#         elif feature == 'Electronic Check':

#             recommendations.append(
#                 "Consider encouraging the customer to switch to an automatic payment method."
#             )


#         elif feature in ["Streaming TV", "Streaming Movies"]:

#             recommendations.append(
#                 "Consider offering a bundled entertainment package based on the customer's streaming usage."
#             )


#         elif feature == 'New Customer':

#             recommendations.append(
#                 "Strengthen early-stage engagement through onboarding support and proactive follow-ups."
#             )


#     recommendations = list(
#         dict.fromkeys(recommendations)
#     )


#     print("===================================")
#     print("       CUSTOMER CHURN ANALYSIS")
#     print("===================================")

#     print(
#         f"Churn Probability: "
#         f"{churn_probability:.2%}"
#     )

#     print(
#         f"Prediction: "
#         f"{'Churn' if prediction == 1 else 'No Churn'}"
#     )

#     print(
#         f"Risk Level: {risk_level}"
#     )


#     print("\n🔴 Top Churn Risk Factors")

#     for _, row in risk_factors.iterrows():

#         print(
#             f"- {row['Feature']} "
#             f"(SHAP: {row['SHAP Value']:.3f})"
#         )


#     print("\n🟢 Top Protective Factors")

#     for _, row in protective_factors.iterrows():

#         print(
#             f"- {row['Feature']} "
#             f"(SHAP: {row['SHAP Value']:.3f})"
#         )


#     print("\n💡 Recommended Actions")

#     for i, recommendation in enumerate(
#         recommendations,
#         start=1
#     ):

#         print(
#             f"{i}. {recommendation}"
#         )


#     return {
#         'churn_probability': churn_probability,
#         'prediction': prediction,
#         'risk_level': risk_level,
#         'risk_factors': risk_factors,
#         'protective_factors': protective_factors,
#         'recommendations': recommendations
#     }

In [ ]:
# import sys
# project_root = os.path.abspath("..")
# sys.path.append(project_root)

In [ ]:
# from utils.model_utils import predict_customer
# from utils.shap_utils import get_shap_explanation
# from utils.recommendation_engine import generate_recommendations

# # Select one customer
# customer = X_test.iloc[[0]]

# # Prediction
# prediction_result = predict_customer(customer)

# # SHAP explanation
# shap_result = get_shap_explanation(customer)

# # Calculate risk level
# probability = prediction_result["churn_probability"]

# if probability >= 0.70:
#     risk_level = "HIGH"
# elif probability >= 0.40:
#     risk_level = "MEDIUM"
# else:
#     risk_level = "LOW"

# # Recommendations
# recommendations = generate_recommendations(
#     probability,
#     shap_result["risk_factors"]
# )

# # Create the complete result dictionary
# result = {
#     "prediction": prediction_result["prediction"],
#     "churn_probability": probability,
#     "risk_level": risk_level,
#     "risk_factors": shap_result["risk_factors"],
#     "protective_factors": shap_result["protective_factors"],
#     "recommendations": recommendations
# }


In [ ]:
# from utils.gemini_utils import generate_ai_summary

# summary = generate_ai_summary(result)

# print(summary)

In [ ]:
# def generate_ai_summary(result):

#     # Get prediction details
#     churn_probability = result['churn_probability']
#     prediction = result['prediction']
#     risk_level = result['risk_level']

#     # Get risk factors
#     risk_factors = [
#         row['Feature']
#         for _, row in result['risk_factors'].iterrows()
#     ]

#     # Get protective factors
#     protective_factors = [
#         row['Feature']
#         for _, row in result['protective_factors'].iterrows()
#     ]

#     # Get recommendations
#     recommendations = result['recommendations']


#     # Create prompt
#     prompt = f"""
# You are a professional customer retention analyst.

# Analyze the following customer churn prediction.

# Churn Probability:
# {churn_probability:.2%}

# Prediction:
# {'Churn' if prediction == 1 else 'No Churn'}

# Risk Level:
# {risk_level}

# Top Churn Risk Factors:
# {risk_factors}

# Top Protective Factors:
# {protective_factors}

# Recommended Business Actions:
# {recommendations}

# Write a concise and professional customer risk summary.

# Your response must:

# 1. Clearly state the customer's churn probability and risk level.
# 2. Explain the main factors contributing to the predicted churn risk.
# 3. Explain the main factors reducing the predicted churn risk.
# 4. Summarize the recommended business actions.
# 5. Do not invent any customer information.
# 6. Do not claim that a factor directly causes churn.
# 7. Use phrases such as "contributes to predicted risk" or
#    "is associated with higher predicted churn risk."
# 8. Keep the response between 100 and 150 words.
# """


#     # Send prompt to Gemini
#     response = client.models.generate_content(
#         model="gemini-3.1-flash-lite",
#         contents=prompt
#     )

#     return response.text

In [ ]:
# result = analyze_customer(0)
# ai_summary = generate_ai_summary(result)

# # Step 3: Display AI summary
# print("\n🤖 AI Business Summary")
# print("----------------------")
# print(ai_summary)